# 1 - Importação das Bibliotecas


In [ ]:
from google.colab import drive

import os
import gc
from datetime import datetime
import sys
from math import trunc
import csv

import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LeakyReLU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

import matplotlib.pyplot as plt

from tqdm.notebook import tqdm

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

#from sklearn.svm import SVC

from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split

import pickle

import shap

import joblib


In [ ]:
# Checagem de disponibilidade da GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ {len(gpus)} GPU(s) disponível(is): {[gpu.name for gpu in gpus]}")
else:
    print("⚠️ Nenhuma GPU disponível. Usando CPU.")

⚠️ Nenhuma GPU disponível. Usando CPU.


In [ ]:
# Acesso ao drive pessoal
drive.mount('/content/drive')

Mounted at /content/drive


# Abertura dos dados para encontrar colunas

In [ ]:
DIRETORIO_BASE = "."
# Caminho para o arquivo compactado
CAMINHO_ARQUIVO = f'{DIRETORIO_BASE}/dados/mh1m_balanceadas.npz'

# Carrega os dados com mmap_mode para uso mais leve de memória
dados = np.load(CAMINHO_ARQUIVO, allow_pickle=True)

# Extração das colunas principais
colunas = dados['column_names']

# Gera Ranking Individuais e Média dos Folds

## Para modelo MLP

In [ ]:
# Caminho para o arquivo compactado
modelos = ["mlp"]
threshold = 0.5
grupos = ["intents","permissions", "opcodes", "apicalls", "permissions_opcodes", "todas"]

In [ ]:
for modelo_nome in modelos:

  for nome_grupo in grupos:
    PASTA_DADOS_SHAP = f"{DIRETORIO_BASE}/src_shap/{nome_grupo}/{modelo_nome}_limiar{trunc(threshold*10)}/"
    PASTA_SAIDA = f"{DIRETORIO_BASE}/src_shap/{nome_grupo}/{modelo_nome}_limiar{trunc(threshold*10)}/"
    os.makedirs(PASTA_SAIDA, exist_ok=True)

    importancias_folds = []
    feature_names_ref = None  # referência de ordem das features (definida no fold 0)

    n_folds = 5;
    for fold in tqdm(range(n_folds)):
      caminho_arquivo_shap = os.path.join(PASTA_DADOS_SHAP,f"{nome_grupo}__{modelo_nome}__fold{fold+1}_SHAP.npz")
      print(caminho_arquivo_shap)
      # Carregar os dados SHAP salvos
      shap_values_carregados = np.load(caminho_arquivo_shap, allow_pickle=True)
      print(shap_values_carregados.files)
      # Extrair os dados
      shap_values_values = shap_values_carregados['shap_values']
      shap_values_base_values = shap_values_carregados['base_values']
      shap_values_data = shap_values_carregados['X_explicado']
      shap_values_feature_names = shap_values_carregados['feature_names']

      print("Dados SHAP carregados com sucesso!")
      print(f"Shape values carregados: {shap_values_values.shape}")
      print(f"Base values carregados: {shap_values_base_values.shape}")
      print(f"Data carregados: {shap_values_data.shape}")
      print(f"Quantidade de Features carregadas: {len(shap_values_feature_names)}")
      print("Total de Features: ",len(shap_values_feature_names))
      print("As 5-primeiras features: ",shap_values_feature_names[0:5])

      if fold == 0:
          feature_names_ref = shap_values_feature_names
      else:
          assert np.array_equal(shap_values_feature_names, feature_names_ref), \
              f"Ordem de features divergente no fold {fold+1} — média entre folds seria inválida!"

      # MLP (KernelExplainer, saída escalar): espera-se array 2D (n_amostras, n_features)
      assert shap_values_values.ndim == 2, \
          f"Formato inesperado do SHAP MLP no fold {fold+1}: {shap_values_values.shape} (esperado 2D)"

      #Importância média das features (global).
      importancias = np.abs(shap_values_values).mean(axis=0)

      # # Transforma em Series (liga valores aos nomes das features)
      importances = pd.Series(importancias, index=shap_values_feature_names.tolist())
      # Ordena mantendo os nomes
      importances = importances.sort_values(ascending=False)
      # print(importances.head(20))
      # Seleciona top 20 (ou top k)
      # top_features = importances.head(20).index.tolist()

      nome_arquivo = os.path.join(PASTA_SAIDA,f"rank_importancias_{modelo_nome}_shap_fold{fold+1}.csv")
      importances.to_csv(nome_arquivo, header=["valor_importancia_media"], index_label="feature_name")

      importancias_folds.append(importancias)

    importancias_medias = np.mean(importancias_folds, axis=0)  # (n_features,)
    idx_ordem = np.argsort(importancias_medias)[::-1]
    features_ordenadas = np.array(feature_names_ref)[idx_ordem]
    importancias_ordenadas = importancias_medias[idx_ordem]

    df_ranking = pd.DataFrame({
        "feature": features_ordenadas,
        "importance_mean_abs_shap": importancias_ordenadas
    })

    caminho_csv = os.path.join(
        PASTA_SAIDA,
        f"{nome_grupo}__{modelo_nome}__ranking_global_SHAP.csv"
    )

    df_ranking.to_csv(caminho_csv, index=False, encoding="utf-8")





## Para modelo RandomForest

In [ ]:
# Caminho para o arquivo compactado
modelos = ["rf"]
grupos = ["intents","permissions", "opcodes", "apicalls", "permissions_opcodes", "todas"]

In [ ]:
for modelo_nome in modelos:

  for nome_grupo in grupos:
    PASTA_DADOS_SHAP = f"{DIRETORIO_BASE}/src_shap/{nome_grupo}/{modelo_nome}/"
    PASTA_SAIDA = f"{DIRETORIO_BASE}/src_shap/{nome_grupo}/{modelo_nome}/"#PASTA_DADOS_SHAP
    os.makedirs(PASTA_SAIDA, exist_ok=True)

    importancias_folds = []
    feature_names_ref = None  # referência de ordem das features (definida no fold 0)


    n_folds = 5;

    for fold in range(n_folds):
      caminho_arquivo_shap = os.path.join(PASTA_DADOS_SHAP,f"{nome_grupo}__{modelo_nome}__fold{fold+1}_SHAP.npz")
      print(caminho_arquivo_shap)
      # Carregar os dados SHAP salvos
      shap_values_carregados = np.load(caminho_arquivo_shap, allow_pickle=True)
      print(shap_values_carregados.files)
      # Extrair os dados
      shap_values_values = shap_values_carregados['shap_values']
      shap_values_base_values = shap_values_carregados['base_values']
      shap_values_data = shap_values_carregados['X_explicado']
      shap_values_feature_names = shap_values_carregados['feature_names']

      # print("Dados SHAP carregados com sucesso!")
      # print(f"Shape values carregados: {shap_values_values.shape}")
      # print(f"Base values carregados: {shap_values_base_values.shape}")
      # print(f"Data carregados: {shap_values_data.shape}")
      # print(f"Quantidade de Features carregadas: {len(shap_values_feature_names)}")
      # print("Total de Features: ",len(shap_values_feature_names))
      # print("As 5-primeiras features: ",shap_values_feature_names[0:5])

      if fold == 0:
          feature_names_ref = shap_values_feature_names
      else:
          assert np.array_equal(shap_values_feature_names, feature_names_ref), \
              f"Ordem de features divergente no fold {fold+1} — média entre folds seria inválida!"

      # RF (classificador binário): espera-se array 3D (n_amostras, n_features, n_classes)
      assert shap_values_values.ndim == 3 and shap_values_values.shape[2] == 2, \
          f"Formato inesperado do SHAP RF no fold {fold+1}: {shap_values_values.shape} (esperado 3D com 2 classes)"


      #Importância média das features (global).
      # ==== Para apenas a classe positiva
      vals = shap_values_values[:, :, 1]  # pega a classe 1
      importancias = np.abs(vals).mean(axis=0) # Usa apenas uma classe

      # === Para a media das duas classes
      # importancias = np.abs(shap_values_values).mean(axis=(0, 2)) # Usa ambas as classes

      importancias_folds.append(importancias)

      # Transforma em Series (liga valores aos nomes das features)
      importances = pd.Series(importancias, index=shap_values_feature_names.tolist())

      # Ordena mantendo os nomes
      importances = importances.sort_values(ascending=False)
      print(importances.head(20))

      # Seleciona top 20 (ou top k)
      # top_features = importances.head(20).index.tolist()

      arquivo = os.path.join(PASTA_SAIDA,f"rank_importancias_{modelo_nome}_shap_fold{fold+1}.csv")
      importances.to_csv(arquivo, header=["valor_importancia_media"], index_label="feature_name")


    importancias_medias = np.mean(importancias_folds, axis=0)  # (n_features,)
    idx_ordem = np.argsort(importancias_medias)[::-1]
    features_ordenadas = np.array(feature_names_ref)[idx_ordem]
    importancias_ordenadas = importancias_medias[idx_ordem]


    df_ranking = pd.DataFrame({
        "feature": features_ordenadas,
        "importance_mean_abs_shap": importancias_ordenadas
    })

    caminho_csv = os.path.join(
        PASTA_SAIDA,
        f"{nome_grupo}__{modelo_nome}__ranking_global_SHAP.csv"
    )

    df_ranking.to_csv(caminho_csv, index=False, encoding="utf-8")




## Para modelo XGBoost

In [ ]:
# Caminho para o arquivo compactado
modelos = ["xgb"]
grupos = ["intents","permissions", "opcodes", "apicalls", "permissions_opcodes", "todas"]

In [ ]:
for modelo_nome in modelos:

  for nome_grupo in grupos:
    PASTA_DADOS_SHAP = f"{DIRETORIO_BASE}/src_shap/{nome_grupo}/{modelo_nome}/"
    PASTA_SAIDA = f"{DIRETORIO_BASE}/src_shap/{nome_grupo}/{modelo_nome}/" #PASTA_DADOS_SHAP
    os.makedirs(PASTA_SAIDA, exist_ok=True)

    importancias_folds = []
    feature_names_ref = None

    n_folds = 5;
    for fold in range(n_folds):
      caminho_arquivo_shap = os.path.join(PASTA_DADOS_SHAP,f"{nome_grupo}__{modelo_nome}__fold{fold+1}_SHAP.npz")
      print(caminho_arquivo_shap)

      shap_values_carregados = np.load(caminho_arquivo_shap, allow_pickle=True)
      print(shap_values_carregados.files)

      shap_values_values = shap_values_carregados['shap_values']
      shap_values_base_values = shap_values_carregados['base_values']
      shap_values_data = shap_values_carregados['X_explicado']
      shap_values_feature_names = shap_values_carregados['feature_names']


      if fold == 0:
          feature_names_ref = shap_values_feature_names
      else:
          assert np.array_equal(shap_values_feature_names, feature_names_ref), \
              f"Ordem de features divergente no fold {fold+1} — média entre folds seria inválida!"

      assert shap_values_values.ndim == 2, \
          f"Formato inesperado do SHAP XGB no fold {fold+1}: {shap_values_values.shape} (esperado 2D)"


      importancias = np.abs(shap_values_values).mean(axis=0)

      importancias_folds.append(importancias)



      importances = pd.Series(importancias, index=shap_values_feature_names.tolist())


      importances = importances.sort_values(ascending=False)


      arquivo = os.path.join(PASTA_SAIDA,f"rank_importancias_{modelo_nome}_shap_fold{fold+1}.csv")
      importances.to_csv(arquivo, header=["valor_importancia_media"], index_label="feature_name")


    importancias_medias = np.mean(importancias_folds, axis=0) 
    idx_ordem = np.argsort(importancias_medias)[::-1]
    features_ordenadas = np.array(feature_names_ref)[idx_ordem]
    importancias_ordenadas = importancias_medias[idx_ordem]

    df_ranking = pd.DataFrame({
        "feature": features_ordenadas,
        "importance_mean_abs_shap": importancias_ordenadas
    })

    caminho_csv = os.path.join(
        PASTA_SAIDA,
        f"{nome_grupo}__{modelo_nome}__ranking_global_SHAP.csv"
    )

    df_ranking.to_csv(caminho_csv, index=False, encoding="utf-8")



